In [2]:
from db_connection import setup_sakila, save_result_csv

engine = setup_sakila(displaylimit=None)

displaylimit: Value None will be treated as 0 (no limit)

## 1. その他の単一行関数

`IF()`、`NULL関連関数`、`CASE式`は、条件によって異なる値を返すときに使用する。

#### 1) IF()

```sql
IF(条件式, 真の場合の値, 偽の場合の値)
```

In [7]:
%%sql movie_rental_duration_result <<

-- レンタル期間で区分
SELECT
    title,
    rental_duration,
    IF(rental_duration >= 5, '長い', '短い') AS duration_type
FROM film;

[📊 全件の結果を見る](../../sql_study/results/7.15/movie_rental_duration_result.csv)

In [25]:
%%sql movie_type_result <<

-- 映画の上映時間で区分
SELECT
    title,
    length,
    IF(length >= 120, '長編', '一般') AS movie_type  
FROM film
ORDER BY movie_type DESC, length ASC;

[📊 全件の結果を見る](../../sql_study/results/7.15/movie_type_result.csv)

#### 2) NULL関連関数

| 関数 | 説明 | 例 |
|---|---|---|
| `IFNULL(a, b)` | `a`が`NULL`なら`b`を返す | `IFNULL(price, 0)` |
| `NULLIF(a, b)` | `a`と`b`が同じなら`NULL`を返す | `NULLIF(discount, 10)` |
| `COALESCE(a, b, c)` | 最初に見つかった`NULLではない値`を返す | `COALESCE(price, discount, 0)` |
| `IS NULL` | `NULL`の行を検索 | `WHERE price IS NULL` |
| `IS NOT NULL` | `NULLではない`行を検索 | `WHERE discount IS NOT NULL` |

#### IFNULL（カラム, 代替値）

`IFNULL()`は、指定した値が`NULL`の場合に`代替値`を返す。

In [13]:
%%sql

SELECT
    title,
    language_id,
    original_language_id,
    IFNULL(original_language_id, 0) AS IFNULLの値
FROM film
LIMIT 5;

title,language_id,original_language_id,IFNULLの値
ACADEMY DINOSAUR,1,None,0
ACE GOLDFINGER,1,None,0
ADAPTATION HOLES,1,None,0
AFFAIR PREJUDICE,1,None,0
AFRICAN EGG,1,None,0


#### NULLIF（値1, 値2）

`NULLIF()`は、`値1`と`値2`が同じ場合に`NULL`を返す。

In [15]:
%%sql

SELECT
    title,
    rental_rate,
    NULLIF(rental_rate, 4.99) AS result
FROM film
LIMIT 5;

title,rental_rate,result
ACADEMY DINOSAUR,0.99,0.99
ACE GOLDFINGER,4.99,None
ADAPTATION HOLES,2.99,2.99
AFFAIR PREJUDICE,2.99,2.99
AFRICAN EGG,2.99,2.99


#### ※ NULL関連関数の実習

`test`データベースに`NULL`を含むテーブルを作成して、NULL関連関数を確認する。

- NULLを含むデータを使うため、練習用のproductテーブルを作成

In [18]:
%%sql

CREATE TABLE product (
    id INT AUTO_INCREMENT PRIMARY KEY,
    name VARCHAR(30),
    price INT,
    discount INT
);

++
||
++
++

- priceまたはdiscountにNULLを含むサンプルデータを追加

In [19]:
%%sql

INSERT INTO product(name, price, discount)
VALUES
('ノートパソコン', 1500000, 10),
('マウス', 30000, NULL),
('キーボード', NULL, 15),
('モニター', 250000, NULL),
('スピーカー', NULL, NULL);

++
||
++
++

- productテーブルのすべてのカラム・データを確認

In [32]:
%%sql

SELECT *
FROM product;

id,name,price,discount
1,ノートパソコン,1500000,10
2,マウス,30000,None
3,キーボード,None,15
4,モニター,250000,None
5,スピーカー,None,None


- priceがNULLの場合、0に置き換えて表示

In [20]:
%%sql

SELECT
    name,
    IFNULL(price, 0) AS price
FROM product;

name,price
ノートパソコン,1500000
マウス,30000
キーボード,0
モニター,250000
スピーカー,0


- discountが10の場合はNULLに変換し、それ以外は元の値を返す

In [33]:
%%sql

SELECT 
    name,
    NULLIF(discount, 10) AS discount
FROM product;

name,discount
ノートパソコン,None
マウス,None
キーボード,15
モニター,None
スピーカー,None


- price → discount → 0 の順に、最初にNULLでない値を表示

In [22]:
%%sql

SELECT
    name,
    COALESCE(price, discount, 0) AS value
FROM product;

name,value
ノートパソコン,1500000
マウス,30000
キーボード,15
モニター,250000
スピーカー,0


- priceがNULLのデータだけを抽出

In [23]:
%%sql

SELECT *
FROM product
WHERE price IS NULL;

id,name,price,discount
3,キーボード,None,15
5,スピーカー,None,None


- discountがNULLではないデータだけを抽出

In [24]:
%%sql

SELECT *
FROM product
WHERE discount IS NOT NULL;

id,name,price,discount
1,ノートパソコン,1500000,10
3,キーボード,None,15


## 2. CASE式

複数の条件を処理するときに使用する。

```sql
CASE
    WHEN 条件1 THEN 結果1
    WHEN 条件2 THEN 結果2
    ELSE 結果3
END
```

※ `IF()`でも複数条件を処理できるが、入れ子が増えると可読性が低下する。

```sql
SELECT
    IF(name = 'ノートパソコン', 1,
        IF(name = 'マウス', 2, 3))
FROM product;
```

### # IF()とCASEの違い

| `IF()` | `CASE` |
|---|---|
| 条件が少ない場合に適している | 複数条件の処理に適している |
| 構文が簡単 | 可読性が高い |
| MySQL専用関数 | 標準SQLの式 |
| 1つの条件の真・偽を判断| `WHEN`を複数記述できる |

In [28]:
%%sql

-- 上映時間で区分
SELECT
    title,
    length,
    CASE
        WHEN length >= 150 THEN '非常に長い'
        WHEN length >= 120 THEN '長編'
        WHEN length >= 90 THEN '普通'
        ELSE '短編'
    END AS length_type
FROM film
LIMIT 6;

title,length,length_type
ACADEMY DINOSAUR,86,短編
ACE GOLDFINGER,48,短編
ADAPTATION HOLES,50,短編
AFFAIR PREJUDICE,117,普通
AFRICAN EGG,130,長編
AGENT TRUMAN,169,非常に長い


In [41]:
%%sql

-- レンタル料金で区分
SELECT
    title,
    rental_rate,
    CASE
        WHEN rental_rate = 0.99 THEN '低価格'
        WHEN rental_rate = 2.99 THEN '中価格'
        WHEN rental_rate = 4.99 THEN '高価格'
        ELSE 'その他'
    END AS price_type
FROM film
LIMIT 6;

title,rental_rate,price_type
ACADEMY DINOSAUR,0.99,低価格
ACE GOLDFINGER,4.99,高価格
ADAPTATION HOLES,2.99,中価格
AFFAIR PREJUDICE,2.99,中価格
AFRICAN EGG,2.99,中価格
AGENT TRUMAN,2.99,中価格


In [43]:
%%sql

-- 映画のレーティングで区分
SELECT
    title,
    rating,
    CASE rating
        WHEN 'G' THEN '全年齢対象'
        WHEN 'PG' THEN '保護者の指導を推奨'
        WHEN 'PG-13' THEN '13歳以上'
        WHEN 'R' THEN '青少年は制限'
        WHEN 'NC-17' THEN '17歳以上'
        ELSE 'その他'
    END AS rating_name
FROM film
LIMIT 6;

title,rating,rating_name
ACADEMY DINOSAUR,PG,保護者の指導を推奨
ACE GOLDFINGER,G,全年齢対象
ADAPTATION HOLES,NC-17,17歳以上
AFFAIR PREJUDICE,G,全年齢対象
AFRICAN EGG,G,全年齢対象
AGENT TRUMAN,PG,保護者の指導を推奨


## 3. WHERE句でIF()の使用例

In [45]:
%%sql

-- 上映時間が`120分以上`の場合は`rental_rate >= 2.99`
-- `120分未満`の場合は`rental_rate >= 0.99`の映画を検索する。

SELECT
    title,
    length,
    rental_rate
FROM film
WHERE IF(
    length >= 120,
    rental_rate >= 2.99,
    rental_rate >= 0.99
)
LIMIT 5;

title,length,rental_rate
ACADEMY DINOSAUR,86,0.99
ACE GOLDFINGER,48,4.99
ADAPTATION HOLES,50,2.99
AFFAIR PREJUDICE,117,2.99
AFRICAN EGG,130,2.99


In [47]:
%%sql

-- `rating`が`G`の場合は`90分以上`、それ以外の場合は`120分以上`の映画を検索する。

SELECT
    title,
    rating,
    length
FROM film
WHERE IF(
    rating = 'G',
    length >= 90,
    length >= 120
)
LIMIT 5;

title,rating,length
AFFAIR PREJUDICE,G,117
AFRICAN EGG,G,130
AGENT TRUMAN,PG,169
ALAMO VIDEOTAPE,G,126
ALASKA PHANTOM,PG,136


In [44]:
%%sql

-- `original_language_id`が`NULL`の場合、そのままでは比較演算子を使用できないため、`0`に置き換えて比較する。

SELECT
    title,
    original_language_id
FROM film
WHERE IFNULL(original_language_id, 0) <= 10 
LIMIT 5;

title,original_language_id
ACADEMY DINOSAUR,None
ACE GOLDFINGER,None
ADAPTATION HOLES,None
AFFAIR PREJUDICE,None
AFRICAN EGG,None


`WHERE IFNULL(original_language_id, 0) = 0`は、次と同じ意味になる。

```sql
WHERE original_language_id = 0
   OR original_language_id IS NULL;
```

#### ※`NULL`は`=`や`IN`では判定せず、`IS NULL`を使用する。

In [54]:
%%sql

SELECT
    title,
    rental_rate
FROM film
WHERE NULLIF(rental_rate, 4.99) IS NULL
LIMIT 5;

title,rental_rate
ACE GOLDFINGER,4.99
AIRPLANE SIERRA,4.99
AIRPORT POLLOCK,4.99
ALADDIN CALENDAR,4.99
ALI FOREVER,4.99


> ! 注意：次のような書き方は正しくない。

```sql
WHERE original_language_id IN (NULL, 0);
```

#### ※ IS NULL / IS NOT NULL 演算子

In [57]:
%%sql

SELECT *
FROM film
WHERE original_language_id IS NULL
LIMIT 3;

film_id,title,description,release_year,language_id,original_language_id,rental_duration,rental_rate,length,replacement_cost,rating,special_features,last_update
1,ACADEMY DINOSAUR,A Epic Drama of a Feminist And a Mad Scientist who must Battle a Teacher in The Canadian Rockies,2006,1,None,6,0.99,86,20.99,PG,"Deleted Scenes,Behind the Scenes",2006-02-15 05:03:42
2,ACE GOLDFINGER,A Astounding Epistle of a Database Administrator And a Explorer who must Find a Car in Ancient China,2006,1,None,3,4.99,48,12.99,G,"Trailers,Deleted Scenes",2006-02-15 05:03:42
3,ADAPTATION HOLES,A Astounding Reflection of a Lumberjack And a Car who must Sink a Lumberjack in A Baloon Factory,2006,1,None,7,2.99,50,18.99,NC-17,"Trailers,Deleted Scenes",2006-02-15 05:03:42


In [58]:
%%sql

SELECT *
FROM film
WHERE original_language_id IS NOT NULL
LIMIT 3;

film_id,title,description,release_year,language_id,original_language_id,rental_duration,rental_rate,length,replacement_cost,rating,special_features,last_update


### # WHERE句でCASEの使用例

In [60]:
%%sql

-- `rating`が`G`の場合は`90分以上`、それ以外の場合は`120分以上`の映画を検索する。

SELECT
    title,
    rating,
    length
FROM film
WHERE
    CASE
        WHEN rating = 'G'
            THEN length >= 90
        ELSE
            length >= 120
    END
LIMIT 5;

title,rating,length
AFFAIR PREJUDICE,G,117
AFRICAN EGG,G,130
AGENT TRUMAN,PG,169
ALAMO VIDEOTAPE,G,126
ALASKA PHANTOM,PG,136


## 4. ORDER BY句でIF()の使用例

- `length`を昇順で並べ替え

In [ ]:
%%sql

SELECT
    title,
    length
FROM film
ORDER BY length ASC
LIMIT 10;

title,length
RIDGEMONT SUBMARINE,46
IRON MOON,46
ALIEN CENTER,46
LABYRINTH LEAGUE,46
KWAI HOMEWARD,46
DOWNHILL ENOUGH,47
HALLOWEEN NUTS,47
HANOVER GALAXY,47
DIVORCE SHINING,47
HAWK CHILL,47


- `IF()`で**120分未満**と**120分以上**に分け、各グループ内では**title**を昇順で並べ替え

In [ ]:
%%sql order_by_flag_result <<

SELECT
    title,
    length,
    IF(length < 120, 0, 1) AS flag -- 計算結果（`0`または`1`）をカラムとして出力する
FROM film
ORDER BY flag ASC, title ASC


[📊 全件の結果を見る](../../sql_study/results/7.15/order_by_flag_result.csv)

#### ※ NULL値を最後に並べ替える

昇順で並べ替えると`NULL`が最初に表示されるため、`NULL`を大きな値に置き換えて最後に並べることができる。

- `discount`をそのまま昇順で並べ替え、NULLが最初に表示されることを確認

In [ ]:
%%sql

SELECT *
FROM product
ORDER BY discount asc;

id,name,price,discount
2,マウス,30000,None
4,モニター,250000,None
5,スピーカー,None,None
1,ノートパソコン,1500000,10
3,キーボード,None,15


- `IFNULL()`でNULLを999として扱い、NULLを最後に並べ替える

In [11]:
%%sql

SELECT *
FROM product
ORDER BY IFNULL(discount, 999) ASC;

id,name,price,discount
1,ノートパソコン,1500000,10
3,キーボード,None,15
2,マウス,30000,None
4,モニター,250000,None
5,スピーカー,None,None


#### ※ CASE式を使った独自順序での並べ替え

- カラムの値を単純な昇順・降順ではなく、`指定した優先順位で並べ替える`ことができる。

In [23]:
%%sql case_designation_order <<

-- `rating`はENUM型で、通常は定義された順序（G → PG → PG-13 → R → NC-17）で扱われる。
-- `CASE`を使うと、例えば "R → PG" → その他 の順に並べ替えられる。

SELECT
    title,
    rating
FROM film
ORDER BY 
    CASE
        WHEN rating = 'R' THEN 1
        WHEN rating = 'PG' THEN 2
        ELSE 3
    END;

[📊 全件の結果を見る](../../sql_study/results/7.15/case_designation_order.csv)

In [48]:
%%sql case_designation_order2 <<

-- 複数の条件を指定することもできる。
-- 以下は"長い映画を先に並べ、同じグループ内ではタイトル順"に並べ替える例。

SELECT
    title,
    length,
    CASE
        WHEN length >= 150 THEN 1
        WHEN length >= 100 THEN 2
        ELSE 3
    END AS priority -- CASE式で設定した優先順位（1・2・3）表示
FROM film
ORDER BY
    priority,
    title;

[📊 全件の結果を見る](../../sql_study/results/7.15/case_designation_order2.csv)